# DeepSeek-Math-7B IFEval-only Evaluation

This notebook runs **IFEval only** for the official DeepSeek-Math models and automatically saves generation checkpoints, official evaluator files, metrics, summaries, and CSV tables to the Colab output directory.

Default setting:
- `MODEL_KEYS_TO_RUN = ["rl"]`
- `RUN_IFEVAL = True`
- `RUN_IFEVAL_OFFICIAL = True`
- `IFEVAL_LIMIT = 300`

CommonsenseQA is intentionally removed/skipped because it has already been run.


## Step 0: Install dependencies

Run this first in Colab. If you already installed them, you can skip or rerun safely.


In [1]:
!pip install -q --upgrade transformers accelerate datasets evaluate sentencepiece tqdm protobuf safetensors
!pip install -q --upgrade bitsandbytes
!pip install -q absl-py immutabledict nltk langdetect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 142.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 51.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-bigtable 2.36.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1

## Step 1: Imports and configuration

Current default: **only run IFEval for DeepSeek-Math-7B-RL**.

Important outputs are saved under:

```python
/content/deepseek_math_eval_results
```

Each model directory will contain:
- `ifeval_generations_300.jsonl` checkpoint, saved after every example
- `ifeval_generations_300.json` final generation file
- `ifeval_input.jsonl` and `ifeval_response.jsonl` for the official evaluator
- `ifeval_official/eval_results_strict.jsonl`
- `ifeval_official_metrics_300.json`
- `summary.json`

Rerunning the notebook is safe: the IFEval generation cell resumes from the existing `.jsonl` checkpoint and skips completed examples.


In [2]:
from pathlib import Path
import os
import gc
import re
import json
from typing import Any, Dict, Optional, List

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# -----------------------------
# Models
# -----------------------------
MODEL_CONFIGS = {
    "sft": {
        "display_name": "DeepSeek-Math-7B-Instruct",
        "model_name": "deepseek-ai/deepseek-math-7b-instruct",
        "stage": "SFT / Instruct",
        "output_dir": "deepseek_math_7b_instruct_results",
    },
    "rl": {
        "display_name": "DeepSeek-Math-7B-RL",
        "model_name": "deepseek-ai/deepseek-math-7b-rl",
        "stage": "RL / GRPO",
        "output_dir": "deepseek_math_7b_rl_results",
    },
}

# -----------------------------
# What to run
# -----------------------------
# Default: CommonsenseQA is already done, so run only IFEval for RL.
# Change to ["sft"] or ["sft", "rl"] if needed.
MODEL_KEYS_TO_RUN = ["rl"]
RUN_IFEVAL = True
RUN_IFEVAL_OFFICIAL = True

# -----------------------------
# Evaluation limit
# -----------------------------
# IFEval train has 541 examples. Use 300 to match your current experiment setup.
# Use None for the full 541 examples.
IFEVAL_LIMIT = 300

# For quick testing, uncomment:
# IFEVAL_LIMIT = 20

# -----------------------------
# Generation settings
# -----------------------------
IFEVAL_MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0

# Print progress every N examples instead of using tqdm.
# This avoids repeated progress-bar lines in Colab / Jupyter output.
PROGRESS_EVERY = 25

# -----------------------------
# Loading settings
# -----------------------------
USE_8BIT = True            # Recommended for Colab T4/L4.
USE_FLOAT16 = True         # Used only if USE_8BIT = False.
TRUST_REMOTE_CODE = True

# DeepSeek-Math instruction format. Use the same wrapper for SFT and RL.
USE_DEEPSEEK_USER_ASSISTANT_WRAPPER = True

# Save directly in Colab's /content when running on Colab.
# If you run locally, it falls back to a relative directory.
BASE_OUTPUT_DIR = Path("/content/deepseek_math_eval_results") if Path("/content").exists() else Path("deepseek_math_eval_results")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Will run models:", MODEL_KEYS_TO_RUN)
print("RUN_IFEVAL:", RUN_IFEVAL)
print("RUN_IFEVAL_OFFICIAL:", RUN_IFEVAL_OFFICIAL)
print("IFEVAL_LIMIT:", IFEVAL_LIMIT)
print("Output base dir:", BASE_OUTPUT_DIR.resolve())


Will run models: ['rl']
RUN_IFEVAL: True
RUN_IFEVAL_OFFICIAL: True
IFEVAL_LIMIT: 300
Output base dir: /content/deepseek_math_eval_results


## Step 2: Model loading and generation helpers

In [3]:
def load_deepseek_model(model_name: str):
    """Load tokenizer and model. 8-bit mode is recommended on Colab."""
    print(f"Loading {model_name} ...")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=TRUST_REMOTE_CODE,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    if USE_8BIT:
        bnb_config = BitsAndBytesConfig(load_in_8bit=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=TRUST_REMOTE_CODE,
            low_cpu_mem_usage=True,
        )
    else:
        dtype = torch.float16 if USE_FLOAT16 and torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype,
            trust_remote_code=TRUST_REMOTE_CODE,
            low_cpu_mem_usage=True,
        )

    model.eval()
    print("Loaded:", model_name)
    return tokenizer, model


def format_for_deepseek(task_prompt: str) -> str:
    """Wrap task prompt for DeepSeek-Math Instruct/RL models."""
    if USE_DEEPSEEK_USER_ASSISTANT_WRAPPER:
        return f"User: {task_prompt}\nAssistant:"
    return task_prompt


def get_first_device(model):
    return next(model.parameters()).device


def generate_text(
    tokenizer,
    model,
    task_prompt: str,
    max_new_tokens: int = 128,
    temperature: float = 0.0,
) -> str:
    """Generate text from a task prompt and return only newly generated tokens."""
    prompt = format_for_deepseek(task_prompt)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False,
    )

    first_device = get_first_device(model)
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    do_sample = temperature > 0

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    # Important: decode only the newly generated part.
    # This avoids returning the whole prompt and avoids parser bugs.
    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def unload_model(tokenizer=None, model=None):
    """Free GPU memory between models."""
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Freed memory.")


# Part A: IFEval dataset

CommonsenseQA has been removed from this version. This notebook only loads and evaluates IFEval.


In [4]:
# No CommonsenseQA loading in this IFEval-only notebook.
print("CommonsenseQA skipped: this notebook only runs IFEval.")


CommonsenseQA skipped: this notebook only runs IFEval.


In [5]:
# CommonsenseQA helper functions are intentionally omitted.
# Your previous CSQA outputs can stay in their old results directory; this notebook will not overwrite them.


# Part B: IFEval generation

IFEval is not a normal classification benchmark. We first generate model responses, then run the official evaluator.


In [6]:
ifeval_full = load_dataset("google/IFEval", split="train")

if IFEVAL_LIMIT is not None:
    ifeval = ifeval_full.select(range(min(IFEVAL_LIMIT, len(ifeval_full))))
else:
    ifeval = ifeval_full

print(ifeval)
print(ifeval[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ifeval_input_data.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

Dataset({
    features: ['key', 'prompt', 'instruction_id_list', 'kwargs'],
    num_rows: 300
})
{'key': 1000, 'prompt': 'Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*.', 'instruction_id_list': ['punctuation:no_comma', 'detectable_format:number_highlighted_sections', 'length_constraints:number_words'], 'kwargs': [{'num_highlights': None, 'relation': None, 'num_words': None, 'num_placeholders': None, 'prompt_to_repeat': None, 'num_bullets': None, 'section_spliter': None, 'num_sections': None, 'capital_relation': None, 'capital_frequency': None, 'keywords': None, 'num_paragraphs': None, 'language': None, 'let_relation': None, 'letter': None, 'let_frequency': None, 'end_phrase': None, 'forbidden_words': None, 'keyword': None, 'frequenc

In [7]:
def ifeval_size_tag() -> str:
    return "full_541" if IFEVAL_LIMIT is None else str(len(ifeval))


def generate_ifeval(tokenizer, model, model_info: Dict[str, str], output_dir: Path) -> Path:
    """Generate IFEval responses with JSONL checkpointing and resume support.

    If Colab disconnects, rerun the notebook/cell. Existing rows in the JSONL
    file will be loaded and skipped. At the end, a normal JSON file is written for
    the official IFEval evaluator.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    tag = ifeval_size_tag()
    jsonl_file = output_dir / f"ifeval_generations_{tag}.jsonl"
    gen_file = output_dir / f"ifeval_generations_{tag}.json"

    done_indices = set()
    generation_by_index = {}

    # Load partial JSONL checkpoint if it exists.
    if jsonl_file.exists():
        with open(jsonl_file, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)
                original_index = item["original_index"]
                generation_by_index[original_index] = item
                done_indices.add(original_index)
        print(f"Found existing IFEval checkpoint: {len(done_indices)} examples in {jsonl_file}")

    total = len(ifeval)
    print(f"IFEval generation: {model_info['display_name']} | total={total} | already_done={len(done_indices)}")

    with open(jsonl_file, "a", encoding="utf-8") as fout:
        for idx, ex in enumerate(ifeval):
            if idx in done_indices:
                continue

            current = idx + 1
            if current == 1 or current % PROGRESS_EVERY == 0 or current == total:
                print(f"IFEval progress: {current}/{total}", flush=True)

            prompt = ex["prompt"]

            response = generate_text(
                tokenizer,
                model,
                prompt,
                max_new_tokens=IFEVAL_MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
            )

            item = {
                "original_index": idx,
                "key": ex.get("key", idx),
                "prompt": prompt,  # Keep original IFEval prompt here for official evaluator.
                "instruction_id_list": ex["instruction_id_list"],
                "kwargs": ex["kwargs"],
                "response": response,
            }

            fout.write(json.dumps(item, ensure_ascii=False) + "\n")
            fout.flush()
            generation_by_index[idx] = item
            done_indices.add(idx)

    generations = [generation_by_index[i] for i in sorted(generation_by_index)]

    with open(gen_file, "w", encoding="utf-8") as f:
        json.dump(generations, f, ensure_ascii=False, indent=2)

    print("Saved checkpoint JSONL:", jsonl_file)
    print("Saved evaluator JSON:", gen_file)
    print("Num examples:", len(generations))
    if generations:
        print("Example response:")
        print(generations[0]["response"][:1000])

    return gen_file


# Part C: Prepare and run official IFEval evaluator

This uses Google's official `instruction_following_eval` script from `google-research`.


In [8]:
# Clone only once.
if not Path("google-research").exists():
    !git clone --depth 1 https://github.com/google-research/google-research.git
else:
    print("google-research already exists.")

Cloning into 'google-research'...
remote: Enumerating objects: 23308, done.
remote: Counting objects: 100% (23308/23308), done.
remote: Compressing objects: 100% (18089/18089), done.
remote: Total 23308 (delta 4694), reused 14612 (delta 4032), pack-reused 0 (from 0)
Receiving objects: 100% (23308/23308), 482.87 MiB | 18.64 MiB/s, done.
Resolving deltas: 100% (4694/4694), done.
Updating files: 100% (22171/22171), done.


In [9]:
def clean_dict(d):
    """Remove keys whose values are None."""
    if d is None:
        return {}
    return {k: v for k, v in d.items() if v is not None}


def normalize_ifeval_kwargs(kwargs, instruction_id_list):
    """
    Convert HF google/IFEval kwargs to official evaluator format.

    Official format:
    kwargs = [
      {... kwargs for instruction 0 ...},
      {... kwargs for instruction 1 ...}
    ]

    Important:
    remove all None values.
    """
    n = len(instruction_id_list)

    if kwargs is None:
        return [{} for _ in range(n)]

    # Case 1: already list-of-dicts
    if isinstance(kwargs, list):
        fixed = []
        for item in kwargs:
            if isinstance(item, dict):
                fixed.append(clean_dict(item))
            else:
                fixed.append({})
        return fixed

    # Case 2: dict-of-lists
    if isinstance(kwargs, dict):
        fixed = [{} for _ in range(n)]

        for key, value in kwargs.items():
            if isinstance(value, list):
                for i in range(min(n, len(value))):
                    if value[i] is not None:
                        fixed[i][key] = value[i]
            else:
                if value is not None:
                    for i in range(n):
                        fixed[i][key] = value

        fixed = [clean_dict(x) for x in fixed]
        return fixed

    return [{} for _ in range(n)]


def write_ifeval_official_files(generation_file: Path, output_dir: Path):
    with open(generation_file, "r", encoding="utf-8") as f:
        generations = json.load(f)

    input_path = output_dir / "ifeval_input.jsonl"
    response_path = output_dir / "ifeval_response.jsonl"

    with open(input_path, "w", encoding="utf-8") as fin, \
         open(response_path, "w", encoding="utf-8") as fout:

        for i, ex in enumerate(generations):
            key = ex.get("key", None)
            if key is None:
                key = ex.get("original_index", i)

            instruction_id_list = ex["instruction_id_list"]
            fixed_kwargs = normalize_ifeval_kwargs(ex["kwargs"], instruction_id_list)

            input_obj = {
                "key": key,
                "instruction_id_list": instruction_id_list,
                "prompt": ex["prompt"],
                "kwargs": fixed_kwargs,
            }

            response_obj = {
                "key": key,
                "prompt": ex["prompt"],
                "response": ex["response"],
            }

            fin.write(json.dumps(input_obj, ensure_ascii=False) + "\n")
            fout.write(json.dumps(response_obj, ensure_ascii=False) + "\n")

    print("Wrote:", input_path)
    print("Wrote:", response_path)
    print("Num examples:", len(generations))
    return input_path, response_path


def read_ifeval_strict_metrics(official_dir: Path) -> Dict[str, float]:
    strict_file = official_dir / "eval_results_strict.jsonl"

    records = []
    with open(strict_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))

    prompt_correct = 0
    instruction_correct = 0
    instruction_total = 0

    for r in records:
        follow_instruction_list = r["follow_instruction_list"]

        if all(follow_instruction_list):
            prompt_correct += 1

        instruction_correct += sum(follow_instruction_list)
        instruction_total += len(follow_instruction_list)

    strict_prompt_accuracy = prompt_correct / len(records)
    strict_instruction_accuracy = instruction_correct / instruction_total

    return {
        "num_examples": len(records),
        "strict_prompt_accuracy": strict_prompt_accuracy,
        "strict_instruction_accuracy": strict_instruction_accuracy,
    }

In [10]:
import subprocess
import shutil


def run_ifeval_official(generation_file: Path, output_dir: Path) -> Dict[str, Any]:
    input_path, response_path = write_ifeval_official_files(generation_file, output_dir)

    official_dir = output_dir / "ifeval_official"
    if official_dir.exists():
        shutil.rmtree(official_dir)
    official_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python",
        "google-research/instruction_following_eval/evaluation_main.py",
        f"--input_data={input_path}",
        f"--input_response_data={response_path}",
        f"--output_dir={official_dir}",
    ]

    print("Running:", " ".join(cmd))

    env = os.environ.copy()
    env["PYTHONPATH"] = "google-research"

    result = subprocess.run(
        cmd,
        env=env,
        text=True,
        capture_output=True,
    )

    print("STDOUT:")
    print(result.stdout)

    print("STDERR:")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"Official IFEval evaluator failed with return code {result.returncode}"
        )

    metrics = read_ifeval_strict_metrics(official_dir)

    tag = ifeval_size_tag()
    metrics_path = output_dir / f"ifeval_official_metrics_{tag}.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    # Also save a stable filename for easy loading later.
    stable_metrics_path = output_dir / "ifeval_metrics.json"
    with open(stable_metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    print(json.dumps(metrics, indent=2))
    print("Saved IFEval metrics:", metrics_path)
    print("Saved stable IFEval metrics:", stable_metrics_path)
    return metrics


# Part D: Run IFEval-only evaluation

Current default: run **only DeepSeek-Math-7B-RL IFEval** and skip CommonsenseQA.

To run SFT instead, change in the configuration cell:

```python
MODEL_KEYS_TO_RUN = ["sft"]
```

To run both models:

```python
MODEL_KEYS_TO_RUN = ["sft", "rl"]
```

IFEval supports resume through the checkpoint file `ifeval_generations_300.jsonl`.


In [11]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [12]:
all_summaries = []

for model_key in MODEL_KEYS_TO_RUN:
    model_info = MODEL_CONFIGS[model_key]
    model_name = model_info["model_name"]
    output_dir = BASE_OUTPUT_DIR / model_info["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 100)
    print("Evaluating:", model_info["display_name"])
    print("Model:", model_name)
    print("Output dir:", output_dir)
    print("RUN_IFEVAL:", RUN_IFEVAL)
    print("RUN_IFEVAL_OFFICIAL:", RUN_IFEVAL_OFFICIAL)
    print("#" * 100)

    tokenizer, model = load_deepseek_model(model_name)

    ifeval_metrics = None
    generation_file = None
    if RUN_IFEVAL:
        generation_file = generate_ifeval(
            tokenizer=tokenizer,
            model=model,
            model_info=model_info,
            output_dir=output_dir,
        )

        if RUN_IFEVAL_OFFICIAL:
            ifeval_metrics = run_ifeval_official(
                generation_file=generation_file,
                output_dir=output_dir,
            )
        else:
            print("Skipping official IFEval evaluator because RUN_IFEVAL_OFFICIAL = False")
    else:
        print("Skipping IFEval because RUN_IFEVAL = False")

    summary = {
        "model": model_name,
        "display_name": model_info["display_name"],
        "stage": model_info["stage"],
        "output_dir": str(output_dir),
    }

    if ifeval_metrics is not None:
        summary["ifeval_official"] = {
            "split": "train",
            "num_examples": ifeval_metrics["num_examples"],
            "strict_prompt_accuracy": ifeval_metrics["strict_prompt_accuracy"],
            "strict_instruction_accuracy": ifeval_metrics["strict_instruction_accuracy"],
        }
    elif generation_file is not None:
        summary["ifeval_generation_file"] = str(generation_file)

    summary_path = output_dir / "summary.json"
    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    all_summaries.append(summary)

    print("\nSummary:")
    print(json.dumps(summary, indent=2))
    print("Saved summary:", summary_path)

    unload_model(tokenizer, model)

combined_summary_path = BASE_OUTPUT_DIR / "all_summaries.json"
with open(combined_summary_path, "w", encoding="utf-8") as f:
    json.dump(all_summaries, f, ensure_ascii=False, indent=2)

print("\nSaved combined summary:", combined_summary_path)
print(json.dumps(all_summaries, indent=2))



####################################################################################################
Evaluating: DeepSeek-Math-7B-RL
Model: deepseek-ai/deepseek-math-7b-rl
Output dir: /content/deepseek_math_eval_results/deepseek_math_7b_rl_results
RUN_IFEVAL: True
RUN_IFEVAL_OFFICIAL: True
####################################################################################################
Loading deepseek-ai/deepseek-math-7b-rl ...


config.json:   0%|          | 0.00/626 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Loaded: deepseek-ai/deepseek-math-7b-rl
IFEval generation: DeepSeek-Math-7B-RL | total=300 | already_done=0
IFEval progress: 1/300


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


IFEval progress: 25/300
IFEval progress: 50/300
IFEval progress: 75/300
IFEval progress: 100/300
IFEval progress: 125/300
IFEval progress: 150/300
IFEval progress: 175/300
IFEval progress: 200/300
IFEval progress: 225/300
IFEval progress: 250/300
IFEval progress: 275/300
IFEval progress: 300/300
Saved checkpoint JSONL: /content/deepseek_math_eval_results/deepseek_math_7b_rl_results/ifeval_generations_300.jsonl
Saved evaluator JSON: /content/deepseek_math_eval_results/deepseek_math_7b_rl_results/ifeval_generations_300.json
Num examples: 300
Example response:
ĠRaymondĠIII,ĠCountĠofĠTripoliĠwasĠaĠrulerĠofĠtheĠTripolitanĠKingdom,ĠaĠprincipalityĠinĠtheĠsouthĠofĠtheĠItalianĠpeninsula.ĠHeĠwasĠtheĠsonĠofĠKingĠRaymondĠIIĠandĠQueenĠMarieĠofĠFrance.ĠHeĠwasĠbornĠinĠ1206ĠinĠtheĠcityĠofĠTripoli.ĠHeĠwasĠtheĠthirdĠsonĠofĠKingĠRaymondĠII,ĠandĠheĠsucceededĠhisĠfatherĠasĠrulerĠofĠtheĠTripolitanĠKingdomĠinĠ1225.ĊRaymondĠIIIĠwasĠknownĠforĠhisĠmilitaryĠcampaignsĠandĠhisĠsupportĠofĠtheĠCrusades.ĠHeĠwasĠinvol

In [13]:
# Optional: rerun only the official evaluator from an existing generation file.
# This is useful if generation already finished but metrics were not saved.
# It is disabled by default to avoid accidentally evaluating an old SFT file.

RUN_EXISTING_GENERATION_EVAL = False

if RUN_EXISTING_GENERATION_EVAL:
    existing_model_key = "rl"  # change to "sft" if needed
    existing_output_dir = BASE_OUTPUT_DIR / MODEL_CONFIGS[existing_model_key]["output_dir"]
    existing_generation_file = existing_output_dir / f"ifeval_generations_{ifeval_size_tag()}.json"

    ifeval_metrics = run_ifeval_official(
        generation_file=existing_generation_file,
        output_dir=existing_output_dir,
    )
else:
    print("Skipped optional existing-generation evaluator cell.")


Skipped optional existing-generation evaluator cell.


# Part E: Final IFEval table

Run this after Part D finishes. It loads saved IFEval metrics and writes a CSV summary to the Colab output directory.


In [14]:
import json
import pandas as pd
from pathlib import Path

model_dirs = [
    BASE_OUTPUT_DIR / "deepseek_math_7b_instruct_results",
    BASE_OUTPUT_DIR / "deepseek_math_7b_rl_results",
]

rows = []

for model_dir in model_dirs:
    if not model_dir.exists():
        print(f"Skip missing model dir: {model_dir}")
        continue

    # Infer model info from directory name.
    if "instruct" in model_dir.name:
        model_name = "DeepSeek-Math-7B-Instruct"
        stage = "SFT / Instruct"
    elif "rl" in model_dir.name:
        model_name = "DeepSeek-Math-7B-RL"
        stage = "RL / GRPO"
    else:
        model_name = model_dir.name
        stage = "Unknown"

    # Load IFEval metrics.
    ifeval_metrics = None
    for candidate in [
        model_dir / f"ifeval_official_metrics_{ifeval_size_tag()}.json",
        model_dir / "ifeval_metrics.json",
    ]:
        if candidate.exists():
            with open(candidate, "r", encoding="utf-8") as f:
                ifeval_metrics = json.load(f)
            print(f"Loaded IFEval metrics: {candidate}")
            break

    # Fallback: compute from official strict JSONL.
    if ifeval_metrics is None:
        strict_file = model_dir / "ifeval_official" / "eval_results_strict.jsonl"

        if strict_file.exists():
            total_prompts = 0
            correct_prompts = 0
            total_instructions = 0
            correct_instructions = 0

            with open(strict_file, "r", encoding="utf-8") as f:
                for line in f:
                    if not line.strip():
                        continue
                    item = json.loads(line)

                    total_prompts += 1
                    if item.get("follow_all_instructions", False):
                        correct_prompts += 1

                    instruction_results = item.get("follow_instruction_list", [])
                    total_instructions += len(instruction_results)
                    correct_instructions += sum(bool(x) for x in instruction_results)

            ifeval_metrics = {
                "num_examples": total_prompts,
                "strict_prompt_accuracy": correct_prompts / total_prompts if total_prompts else None,
                "strict_instruction_accuracy": correct_instructions / total_instructions if total_instructions else None,
            }
            print(f"Computed IFEval from: {strict_file}")

    rows.append({
        "Model": model_name,
        "Stage": stage,
        "IFEval num examples": ifeval_metrics.get("num_examples") if ifeval_metrics else None,
        "IFEval strict prompt acc": ifeval_metrics.get("strict_prompt_accuracy") if ifeval_metrics else None,
        "IFEval strict instruction acc": ifeval_metrics.get("strict_instruction_accuracy") if ifeval_metrics else None,
    })

df = pd.DataFrame(rows)
display(df)

# Add delta only when both SFT and RL rows exist and both have IFEval metrics.
if len(df) >= 2:
    delta = {
        "Model": "Δ RL - SFT",
        "Stage": "Transfer effect",
        "IFEval num examples": "",
        "IFEval strict prompt acc": df.loc[1, "IFEval strict prompt acc"] - df.loc[0, "IFEval strict prompt acc"]
            if pd.notna(df.loc[1, "IFEval strict prompt acc"]) and pd.notna(df.loc[0, "IFEval strict prompt acc"]) else None,
        "IFEval strict instruction acc": df.loc[1, "IFEval strict instruction acc"] - df.loc[0, "IFEval strict instruction acc"]
            if pd.notna(df.loc[1, "IFEval strict instruction acc"]) and pd.notna(df.loc[0, "IFEval strict instruction acc"]) else None,
    }

    df_with_delta = pd.concat([df, pd.DataFrame([delta])], ignore_index=True)
    display(df_with_delta)

    out_csv = BASE_OUTPUT_DIR / "deepseek_ifeval_sft_vs_rl_comparison.csv"
    df_with_delta.to_csv(out_csv, index=False)
    print("Saved:", out_csv)
else:
    out_csv = BASE_OUTPUT_DIR / "deepseek_ifeval_single_model_results.csv"
    df.to_csv(out_csv, index=False)
    print("Saved:", out_csv)

# Also save an Excel-free JSON copy of the table.
out_json = BASE_OUTPUT_DIR / "deepseek_ifeval_results_table.json"
df.to_json(out_json, orient="records", force_ascii=False, indent=2)
print("Saved:", out_json)


Skip missing model dir: /content/deepseek_math_eval_results/deepseek_math_7b_instruct_results
Loaded IFEval metrics: /content/deepseek_math_eval_results/deepseek_math_7b_rl_results/ifeval_official_metrics_300.json


,Model,Stage,IFEval num examples,IFEval strict prompt acc,IFEval strict instruction acc
0,DeepSeek-Math-7B-RL,RL / GRPO,300,0.173333,0.300439


Saved: /content/deepseek_math_eval_results/deepseek_ifeval_single_model_results.csv
Saved: /content/deepseek_math_eval_results/deepseek_ifeval_results_table.json
